# 12 — MCP og Agent-til-Agent-protokoller

**Fase:** 2 — Kjerne AI | **Tid:** 2–3 timer | **Krav:** Notatbok 10, 11

**Hva du bygger:** En MCP-server som eksponerer pensjonsverktøy, og en agent som bruker den — akkurat slik SPK og Avinor bygger sine AI-systemer.

---

## Hva er MCP?

**Model Context Protocol** (lansert av Anthropic, 2024) er en åpen standard for å koble AI-agenter til verktøy og datakilder.

**Problemet det løser:**  
Uten MCP måtte du skrive ny integrasjonskode for hvert verktøy + hvert AI-system.

```
Uten MCP:   Agent A + Verktøy 1 → Tilpasset kode
            Agent A + Verktøy 2 → Tilpasset kode
            Agent B + Verktøy 1 → Tilpasset kode   (x antall kombinasjoner)

Med MCP:    Verktøy → MCP-server (standardformat)
            Agent   → MCP-klient (standardformat)
            Alle agenter snakker med alle verktøy automatisk
```

**Tenk på det som USB-C for AI-verktøy.**

## Arkitektur

```
┌─────────────┐      MCP-protokoll      ┌─────────────────┐
│  AI-agent   │ ◄──────────────────────► │   MCP-server    │
│  (klient)   │   list_tools()           │  (dine verktøy) │
│             │   call_tool(navn, args)  │                 │
└─────────────┘                          └─────────────────┘
```

In [ ]:
%pip install -q mcp openai

---

## Del 1: Bygg en MCP-server

En MCP-server er en prosess som eksponerer verktøy via standardprotokoll.
Vi skriver den til en fil og kjører den separat.

In [ ]:
# Skriv MCP-serveren til en Python-fil
server_kode = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("SPK Pensjonsverktøy")

@mcp.tool()
def hent_pensjonsregler(pensjonstype: str) -> str:
    """
    Hent regler for en pensjonstype.
    pensjonstype: 'AFP', 'alderspensjon', eller 'uforepensjon'
    """
    regler = {
        "afp": "AFP: Tidligpensjon fra 62 år, krever 3 år offentlig tjeneste, livsvarig.",
        "alderspensjon": "Alderspensjon: Fra 67 år, full sats ved 30 opptjeningsår.",
        "uforepensjon": "Uforepensjon: Ved minst 20% varig nedsatt arbeidsevne, sats 66%.",
    }
    return regler.get(pensjonstype.lower(), f"Ukjent pensjonstype: {pensjonstype}")

@mcp.tool()
def beregn_pensjon(aarslonn: int, opptjeningsaar: int) -> str:
    """
    Beregn estimert maanedlig pensjon.
    aarslonn: Aarlig lonn i kroner
    opptjeningsaar: Antall aar i pensjonsordningen
    """
    sats = min(0.66, 0.66 * opptjeningsaar / 30)
    maanedlig = int(aarslonn * sats / 12)
    return f"{maanedlig:,} kr/mnd (sats {sats:.0%}, basert paa {opptjeningsaar} aar)"

@mcp.resource("pensjon://satser")
def hent_satser() -> str:
    """Gjeldende pensjonssatser."""
    return "AFP: 66% | Alderspensjon: 66% | Uforepensjon: 66%"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("spk_mcp_server.py", "w") as f:
    f.write(server_kode)

print("MCP-server skrevet til spk_mcp_server.py")

---

## Del 2: Bruk MCP-klient

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def list_mcp_verktøy():
    """Oppdag alle verktøy en MCP-server tilbyr."""
    server_params = StdioServerParameters(
        command="python",
        args=["spk_mcp_server.py"],
    )
    async with stdio_client(server_params) as (lese, skrive):
        async with ClientSession(lese, skrive) as session:
            await session.initialize()
            verktøy = await session.list_tools()
            print("Tilgjengelige MCP-verktøy:")
            for v in verktøy.tools:
                print(f"  - {v.name}: {v.description}")
            return verktøy.tools

await list_mcp_verktøy()

In [ ]:
async def kall_mcp_verktøy(verktøy_navn: str, argumenter: dict):
    """Kall et spesifikt MCP-verktøy."""
    server_params = StdioServerParameters(
        command="python", args=["spk_mcp_server.py"]
    )
    async with stdio_client(server_params) as (lese, skrive):
        async with ClientSession(lese, skrive) as session:
            await session.initialize()
            resultat = await session.call_tool(verktøy_navn, arguments=argumenter)
            return resultat.content[0].text

# Test verktøykall
regler = await kall_mcp_verktøy("hent_pensjonsregler", {"pensjonstype": "AFP"})
print(f"AFP-regler: {regler}")

beregning = await kall_mcp_verktøy("beregn_pensjon", {"aarslonn": 650000, "opptjeningsaar": 28})
print(f"Beregning: {beregning}")

---

## Del 3: Agent som bruker MCP-server

In [ ]:
import json
from openai import OpenAI

llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

async def agent_med_mcp(spørsmål: str) -> str:
    """
    Agent som oppdager og bruker MCP-verktøy dynamisk.
    I produksjon: verktøylisten hentes fra MCP-server, ikke hardkodet.
    """
    server_params = StdioServerParameters(
        command="python", args=["spk_mcp_server.py"]
    )
    async with stdio_client(server_params) as (lese, skrive):
        async with ClientSession(lese, skrive) as session:
            await session.initialize()

            # Hent verktøyliste fra MCP-server dynamisk
            mcp_verktøy = await session.list_tools()
            openai_verktøy = [
                {
                    "type": "function",
                    "function": {
                        "name": v.name,
                        "description": v.description,
                        "parameters": v.inputSchema,
                    }
                }
                for v in mcp_verktøy.tools
            ]

            meldinger = [
                {"role": "system", "content": "Du er en pensjonsrådgiver. Bruk verktøyene. Svar på norsk."},
                {"role": "user",   "content": spørsmål},
            ]

            for _ in range(5):
                svar = llm.chat.completions.create(
                    model="llama3.2", messages=meldinger,
                    tools=openai_verktøy, tool_choice="auto",
                )
                melding = svar.choices[0].message
                if not melding.tool_calls:
                    return melding.content

                meldinger.append(melding)
                for kall in melding.tool_calls:
                    args = json.loads(kall.function.arguments)
                    res  = await session.call_tool(kall.function.name, arguments=args)
                    meldinger.append({
                        "role": "tool",
                        "tool_call_id": kall.id,
                        "content": res.content[0].text,
                    })
    return "Ingen svar."

svar = await agent_med_mcp(
    "Jeg har 28 år i staten og tjener 680 000 kr. Hva er AFP-reglene og hva vil jeg få?"
)
print(svar)

---

## Del 4: A2A — Agent til Agent

**Agent-to-Agent (A2A)** er en protokoll (Google, 2025) der en agent kan delegere oppgaver til andre agenter — som et API mellom AI-systemer.

In [ ]:
# A2A konseptuelt eksempel (protokollen er ny, biblioteker under utvikling)
# Viser mønsteret — i produksjon: bruk a2a-sdk eller google-a2a

class EnkelAgent:
    """Mini-implementasjon av A2A-mønsteret."""

    def __init__(self, navn: str, beskrivelse: str, ferdigheter: list[str]):
        self.navn        = navn
        self.beskrivelse = beskrivelse
        self.ferdigheter = ferdigheter

    def agent_kort(self) -> dict:
        """A2A Agent Card: forteller andre agenter hva denne kan."""
        return {
            "navn":        self.navn,
            "beskrivelse": self.beskrivelse,
            "ferdigheter": self.ferdigheter,
        }

    def utfør(self, oppgave: str) -> str:
        raise NotImplementedError


class PensjonsAgent(EnkelAgent):
    def utfør(self, oppgave: str) -> str:
        return f"[PensjonsAgent] Svar på '{oppgave}': AFP kan tas fra 62 år."


class OrkestreringsAgent(EnkelAgent):
    def __init__(self, underagenter: list[EnkelAgent]):
        super().__init__("Orkestrering", "Delegerer til spesialiserte agenter", ["orkestrering"])
        self.underagenter = underagenter

    def finn_riktig_agent(self, oppgave: str) -> EnkelAgent:
        """Velg agent basert på ferdigheter (forenklet)."""
        for agent in self.underagenter:
            if any(ord in oppgave.lower() for ord in agent.ferdigheter):
                return agent
        return self.underagenter[0]

    def utfør(self, oppgave: str) -> str:
        valgt = self.finn_riktig_agent(oppgave)
        print(f"  Orkestrering → delegerer til: {valgt.navn}")
        return valgt.utfør(oppgave)


# Sett opp multi-agent-system
pensjon_agent = PensjonsAgent("Pensjon", "Pensjonsregler", ["pensjon", "afp", "uføre"])
orkestrering  = OrkestreringsAgent([pensjon_agent])

print("Agent Cards:")
for agent in [pensjon_agent, orkestrering]:
    print(f"  {agent.agent_kort()}")

print("\nKjør A2A-flyt:")
svar = orkestrering.utfør("Hva er reglene for AFP?")
print(f"  Svar: {svar}")

---

## Oppsummering

| Konsept | Hva det er |
|---------|----------|
| MCP | Standard for å eksponere verktøy til AI-agenter |
| MCP-server | Prosess som tilbyr verktøy via standardprotokoll |
| MCP-klient | Agent som oppdager og bruker MCP-verktøy |
| A2A | Protokoll for agent-til-agent-kommunikasjon |
| Agent Card | A2A: maskinlesbar beskrivelse av en agents evner |

---

## Hva er neste steg?

**Neste: `13_knowledge_graphs.ipynb`** — Når RAG trenger å forstå *relasjoner* mellom konsepter, ikke bare finne lignende tekst. Kunnskapsgrafer + GraphRAG.